# YAML-BERT Training on Colab

Train YAML-BERT with tree positional encoding + kind embedding on a free T4 GPU.

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# Clone the repo
!git clone https://github.com/llms-study/yaml-bert.git
%cd yaml-bert

In [ ]:
# Install dependencies
!pip install -q pyyaml torch datasets matplotlib tqdm scikit-learn

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## Option A: Quick training (50K docs, ~30 min)

In [ ]:
!python scripts/train_hf.py --max-docs 50000 --full --epochs 10 --vocab-min-freq 100 --output-dir output_colab

## Option B: Full training (276K docs)

With T4's 16GB VRAM, we can use larger batch size for faster training.

In [ ]:
!python scripts/train_hf.py --max-docs 0 --full --epochs 15 --vocab-min-freq 100 --batch-size 128 --output-dir output_colab

## Option C: Layer comparison experiment

Train 3 models with different layer counts on 50K docs to find optimal depth.

In [ ]:
import glob
import os
import sys
sys.path.insert(0, '.')

from yaml_bert.config import YamlBertConfig
from yaml_bert.annotator import DomainAnnotator
from yaml_bert.dataset import YamlDataset
from yaml_bert.embedding import YamlBertEmbedding
from yaml_bert.linearizer import YamlLinearizer
from yaml_bert.model import YamlBertModel
from yaml_bert.trainer import YamlBertTrainer
from yaml_bert.evaluate import YamlBertEvaluator
from yaml_bert.vocab import VocabBuilder

# Build vocab and dataset once
linearizer = YamlLinearizer()
annotator = DomainAnnotator()
builder = VocabBuilder()

config_base = YamlBertConfig(d_model=256, num_heads=8)
vocab = builder.build_from_huggingface(
    'substratusai/the-stack-yaml-k8s',
    linearizer=linearizer, annotator=annotator,
    max_docs=50000, min_freq=100,
    counts_path='output_layer_exp/token_counts.json',
)
os.makedirs('output_layer_exp', exist_ok=True)
vocab.save('output_layer_exp/vocab.json')
print(f'Key vocab: {len(vocab.key_vocab)}, Value vocab: {len(vocab.value_vocab)}, Kind vocab: {vocab.kind_vocab_size}')

dataset = YamlDataset.from_huggingface(
    'substratusai/the-stack-yaml-k8s',
    vocab=vocab, linearizer=linearizer, annotator=annotator,
    config=config_base, max_docs=50000,
)
print(f'Dataset: {len(dataset)} docs')

In [ ]:
# Train with different layer counts
results = {}

for num_layers in [2, 4, 6, 8]:
    print(f'\n{"="*60}')
    print(f'Training with {num_layers} layers')
    print(f'{"="*60}')
    
    config = YamlBertConfig(
        d_model=256, num_layers=num_layers, num_heads=8,
        num_epochs=10, batch_size=64,
    )
    
    emb = YamlBertEmbedding(
        config=config,
        key_vocab_size=vocab.key_vocab_size,
        value_vocab_size=vocab.value_vocab_size,
        kind_vocab_size=vocab.kind_vocab_size,
    )
    model = YamlBertModel(
        config=config, embedding=emb,
        key_vocab_size=vocab.key_vocab_size,
    )
    
    num_params = sum(p.numel() for p in model.parameters())
    print(f'Parameters: {num_params:,}')
    
    trainer = YamlBertTrainer(
        config=config, model=model, dataset=dataset,
        checkpoint_dir=f'output_layer_exp/layers_{num_layers}',
        checkpoint_every=10,
    )
    losses = trainer.train()
    
    # Evaluate
    evaluator = YamlBertEvaluator(model=model, dataset=dataset, vocab=vocab)
    accuracy = evaluator.evaluate_prediction_accuracy()
    
    results[num_layers] = {
        'params': num_params,
        'final_loss': losses[-1],
        'top1': accuracy['top1_accuracy'],
        'top5': accuracy['top5_accuracy'],
    }
    print(f'\nLayers={num_layers}: loss={losses[-1]:.4f}, top1={accuracy["top1_accuracy"]:.2%}, top5={accuracy["top5_accuracy"]:.2%}')

In [ ]:
# Compare results
import matplotlib.pyplot as plt

layers = sorted(results.keys())
top1s = [results[l]['top1'] for l in layers]
top5s = [results[l]['top5'] for l in layers]
params = [results[l]['params'] for l in layers]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(layers, top1s, 'bo-', label='Top-1')
ax1.plot(layers, top5s, 'ro-', label='Top-5')
ax1.set_xlabel('Number of Layers')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy vs Layers')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(layers, [p/1e6 for p in params], color='steelblue')
ax2.set_xlabel('Number of Layers')
ax2.set_ylabel('Parameters (M)')
ax2.set_title('Model Size vs Layers')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('output_layer_exp/layer_comparison.png', dpi=150)
plt.show()

print('\nResults:')
print(f'{"Layers":<8} {"Params":<12} {"Loss":<10} {"Top-1":<10} {"Top-5":<10}')
for l in layers:
    r = results[l]
    print(f'{l:<8} {r["params"]:>10,}  {r["final_loss"]:<10.4f} {r["top1"]:<10.2%} {r["top5"]:<10.2%}')

## Evaluate

Run capability tests and anomaly detection on the best checkpoint.

In [ ]:
# Run capability tests
!python model_tests/test_capabilities.py output_colab/checkpoints/yaml_bert_epoch_10.pt

In [ ]:
# Run anomaly detection
!python scripts/anomaly_score.py output_colab/checkpoints/yaml_bert_epoch_10.pt --run-examples

In [ ]:
# Download the checkpoint to local machine
from google.colab import files
files.download('output_colab/checkpoints/yaml_bert_epoch_10.pt')
files.download('output_colab/vocab.json')